# Lab 18-06 · Sampling Inputs, Averaging and a PWM DAC

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import re
import shutil
import subprocess
import tempfile
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import mcuanalog as ma

## Part 1 · Settling

*Predict* the worst-case error (in LSB) of a full-scale step from a 10 kΩ source after 62.5 ns and after 312 ns.

In [2]:
tau = (10e3 + 500) * 3.3e-12
t = np.linspace(0, 400e-9, 401)
plt.semilogy(t * 1e9, 4096 * np.exp(-t / tau)); plt.axhline(0.5, color="grey", linestyle=":")
plt.xlabel("sampling time (ns)"); plt.ylabel("error of a full-scale step (LSB)"); plt.show()
for ts in (62.5e-9, 312e-9):
    print(f"t_s = {ts * 1e9:5.1f} ns: error {4096 * np.exp(-ts / tau):8.2f} LSB")
print("needed for half an LSB:", round(ma.settle_time(10.5e3, 3.3e-12, 12) * 1e9, 1), "ns")

t_s =  62.5 ns: error   674.53 LSB
t_s = 312.0 ns: error     0.50 LSB
needed for half an LSB: 312.2 ns


## Part 2 · The external capacitor

*Predict* the error of Example 18-06.3 at 1, 10 and 100 kHz, then run the ngspice netlist.

In [3]:
for f in (1e3, 10e3, 100e3):
    x, s = ma.mux_equilibrium(3.3, 10e3, 100e-9, 3.3e-12, f)
    print(f"{f / 1e3:6.0f} kHz: recursion {1e3 * (3.3 - s):7.3f} mV, average-current model {1e3 * ma.mux_error_approx(3.3, 10e3, 3.3e-12, f):7.3f} mV")
if shutil.which("ngspice"):
    out = subprocess.run(["ngspice", "-b", "sh_mux.cir"], capture_output=True, text=True).stdout
    print("ngspice:", re.search(r"err_mv\s*=\s*(\S+)", out).group(1), "mV")
else:
    print("ngspice not installed: see SETUP.md")

     1 kHz: recursion   0.172 mV, average-current model   0.109 mV
    10 kHz: recursion   1.144 mV, average-current model   1.089 mV
   100 kHz: recursion  10.908 mV, average-current model  10.854 mV


ngspice: 1.09038e+01 mV


## Part 3 · Averaging

*Predict* the rms error of $M$-sample averages for an input 0.3 LSB above a code, with 0.1, 0.3 and 0.8 LSB of noise.

In [4]:
v_in = (1000 + 0.3) / 4096          # 1 V reference, 12 bits: 0.3 LSB above code 1000
ms = [1, 4, 16, 64, 256]
for sigma in (0.1, 0.3, 0.8):
    rms = [np.sqrt(np.mean((ma.averaged_code(v_in, sigma, m, trials=4000) - 1000.3) ** 2)) for m in ms]
    plt.loglog(ms, rms, "o-", label=f"noise {sigma} LSB rms")
    print(f"sigma {sigma}: " + ", ".join(f"M={m}: {r:.4f}" for m, r in zip(ms, rms)))
plt.loglog(ms, 0.85 / np.sqrt(ms), "k:", label="1/sqrt(M) from 0.85 LSB")
plt.xlabel("samples averaged, M"); plt.ylabel("rms error (LSB)"); plt.legend(); plt.show()

sigma 0.1: M=1: 0.3170, M=4: 0.2868, M=16: 0.2796, M=64: 0.2779, M=256: 0.2775
sigma 0.3: M=1: 0.4424, M=4: 0.2271, M=16: 0.1228, M=64: 0.0753, M=256: 0.0584
sigma 0.8: M=1: 0.8524, M=4: 0.4196, M=16: 0.2141, M=64: 0.1083, M=256: 0.0544


## Part 4 · PWM

*Predict* the ripple of Example 18-06.6's filter at $D$ = 0.5 and 0.1, and what a second buffered RC stage does.

In [5]:
T, tau = 16e-6, 4.096e-3
for d in (0.5, 0.1):
    print(f"D = {d}: exact {1e3 * ma.pwm_ripple_exact(3.3, d, T, tau):.4f} mV, approximation {1e3 * ma.pwm_ripple_approx(3.3, d, T, tau):.4f} mV")
if shutil.which("ngspice"):
    out = subprocess.run(["ngspice", "-b", "pwm_rc.cir"], capture_output=True, text=True).stdout
    print("ngspice ripple:", re.search(r"ripple_mv\s*=\s*(\S+)", out).group(1), "mV")
# one and two buffered stages in steady state, from the Fourier series of the square wave (odd harmonics)
w0 = 2 * np.pi / T
tt = np.linspace(0, T, 2001)
for stages in (1, 2):
    y = np.zeros_like(tt)
    for n in range(1, 20001, 2):
        H = (1 / (1 + 1j * n * w0 * tau)) ** stages
        y += np.abs(H) * (2 * 3.3 / (np.pi * n)) * np.sin(n * w0 * tt + np.angle(H))
    print(f"{stages} stage(s): ripple {1e6 * (y.max() - y.min()):10.3f} uV")
print(f"2 pi f tau = {w0 * tau:.0f}")

D = 0.5: exact 3.2227 mV, approximation 3.2227 mV
D = 0.1: exact 1.1602 mV, approximation 1.1602 mV


ngspice ripple: 3.22265e+00 mV


1 stage(s): ripple   3222.590 uV


2 stage(s): ripple      1.574 uV
2 pi f tau = 1608


## Part 5 · Integer arithmetic

*Predict* the four values the C test prints, then run it.

In [6]:
cc = shutil.which("cc") or shutil.which("clang") or shutil.which("gcc")
if cc:
    with tempfile.TemporaryDirectory() as td:
        subprocess.run([cc, "-std=c17", "-O1", "-o", f"{td}/t", "c/adcmath_test.c", "-lm"], check=True)
        print(subprocess.run([f"{td}/t"], capture_output=True, text=True).stdout)
else:
    print("no C compiler found")

code 1241 at 3300 mV: 999829 uV
bandgap code 1489 of 1200 mV: VDD 3301 mV
eight 2047s and eight 2048s: 14-bit 8190
six 2047s and ten 2048s: 14-bit 8191
TEST PASSED



## Discussion

The questions and worked answers are in the [handout](README.md#discussion).